In [ ]:
import ipywidgets

from tqdm.auto import tqdm
tqdm.pandas()

import os
import sys

module_path = os.path.abspath(os.path.join('..'))
if not module_path in sys.path:
    sys.path.insert(0, module_path)

from innoprod.sheet_tools import get_sheet_dfs
from innoprod.wrangling.wrangling_tools import is_non_empty
from innoprod.wrangling.msyh_data_sharing import wrangle_roadmaps
from innoprod.text_analysis import code_matching
from innoprod.text_analysis.matching_status import MatchingStatus
from innoprod.text_analysis import interactive_matching

In [ ]:
data = get_sheet_dfs()
roadmaps_df = wrangle_roadmaps(data['Roadmaps'])

In [ ]:
cols = [
    'Summary review of Edge Digital diagnostic report & current state and key improvement areas',
    'What are the internal barriers to growth? How do you intend to finance future growth? Are there sufficient leadership and management skills in the business to achieve your growth? What opportunities do you have to expand into new markets?',
    'Level of current Strategic Digital Skills/knowledge in the business',
    'Level of current Technical Digital Skills/knowledge in the business',
    'Whether the business is already investing/adopting/utilising Industry 4.0 Technologies, with examples',
    'Summary of the identified problems, including Gap Analysis',
    'Key potential industry 4.0 solutions to address the identified problems/gaps',
    'Recommended Action Plan to utilise Industry 4.0 Technology',
    'Overview of qualitative benefits of recommended Action Plan (positive/negative)',
    'Skills and other requirements that will be needed to successfully implement the recommended Action Plan',
    'What has been your overall opinion of the support you have received in this programme? (Add comments)'
]

# All question responses as long table

In [ ]:
responses_df = roadmaps_df[['Client ID'] + cols].melt(id_vars=['Client ID'], value_vars=cols, var_name='Question', value_name='Response')
responses_df = responses_df[is_non_empty(responses_df['Response'])]

responses_df

# Break into sentences

In [ ]:
from nltk.tokenize import PunktSentenceTokenizer
tokenizer = PunktSentenceTokenizer()

sentences_df = responses_df.copy()
sentences_df['Sentences'] = sentences_df.apply(lambda row: tokenizer.tokenize(row['Response']), axis=1)
sentences_df = sentences_df[['Client ID', 'Question', 'Sentences']].explode('Sentences').reset_index().rename(columns={'index' : 'Sentence Number', 'Sentences' : 'Sentence'})
sentences_df['Sentence Number'] = sentences_df.groupby('Sentence Number').cumcount() + 1
sentences_df

# Pre-process sentences

In [ ]:
sentences_df['Cleaned Sentence'] = sentences_df['Sentence'].str.lower()

# TODO Standardise acronyms? 
# Need to be very careful if removing stop words, as the standard stop word lists contains words like "not" that are important here.
# Consider lemmatisation/stemming?

# Count repeated sentences

In [ ]:
recurring_sentences_df = sentences_df['Cleaned Sentence'].value_counts().to_frame().reset_index().rename(columns={'count': 'Count'})
# recurring_sentences_df = recurring_sentences_df.merge(sentences_df[['Cleaned Sentence', 'Question']], on='Cleaned Sentence', how='left').drop_duplicates(subset=['Cleaned Sentence', 'Question'])
# recurring_sentences_df = recurring_sentences_df.groupby(['Cleaned Sentence', 'Count'])['Question'].apply(list).to_frame().rename(columns={'Question': 'Questions'}).reset_index().sort_values(by='Count', ascending=False)
# recurring_sentences_df['Num Questions'] = recurring_sentences_df['Questions'].apply(len)
recurring_sentences_df

# Manual iterative pattern matching

In [ ]:
codes_df = None
recurring_sentences_df['Cleaned Sentence'][0]

In [ ]:
code_name = 'Lack of formal digital strategy'
code_patterns = ['development of a formal digital strategy should be the starting point']

codes_df, recurring_sentences_df = code_matching.add_code(code_name, code_patterns, codes_df, recurring_sentences_df)
codes_df

In [ ]:
recurring_sentences_df

In [ ]:
w = interactive_matching.verification_widget(recurring_sentences_df, code_name)
display(w)

In [ ]:
recurring_sentences_df = interactive_matching.apply_verification(recurring_sentences_df, code_name, w)
recurring_sentences_df